In [1]:
import sys
import os

# Adiciona o diretório pai (a raiz do projeto) ao 'sys.path'
# '..' significa "subir um nível"
project_root = os.path.abspath('..')

# Adiciona o caminho apenas se ele ainda não estiver lá
if project_root not in sys.path:
    sys.path.append(project_root)

# Agora esta linha deve funcionar!
from config import settings
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

settings.PIORES_RANKINGS_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.float_format = '{:.2f}'.format

# Leitura dos Dados

In [22]:
df_insta = pd.read_excel(settings.DEPUTADOS_INSTA_XLSX_OUT)
df_insta['Nome Civil'] = df_insta['Nome Civil'].str.title()

df_insta.head()

,Nome Parlamentar,Partido,Telefone,Correio Eletrônico,Nome sem Acento,Tratamento,Nome Civil,Instagram
0,Acácio Favacho,MDB-AP,3215-5414,dep.acaciofavacho@camara.leg.br,Acacio Favacho,Exmo. Senhor Deputado,Acácio Da Silva Favacho Neto,https://www.instagram.com/acaciofavacho/
1,Adail Filho,REPUBLICANOS-AM,3215-5531,dep.adailfilho@camara.leg.br,Adail Filho,Exmo. Senhor Deputado,Adail José Figueiredo Pinheiro,https://www.instagram.com/adailfilho/
2,Adilson Barroso,PL-SP,3215-5750,dep.adilsonbarroso@camara.leg.br,Adilson Barroso,Exmo. Senhor Deputado,Adilson Barroso Oliveira,https://www.instagram.com/adilson_barroso_ambi...
3,Adolfo Viana,PSDB-BA,3215-5911,dep.adolfoviana@camara.leg.br,Adolfo Viana,Exmo. Senhor Deputado,Adolfo Viana De Castro Neto,https://www.instagram.com/adolfovianaoficial/
4,Adriana Ventura,NOVO-SP,3215-5802,dep.adrianaventura@camara.leg.br,Adriana Ventura,Exma. Senhora Deputada,Adriana Miguel Ventura,https://www.instagram.com/adriventurasp/


In [23]:
# Ler o Excel em um DataFrame
df_contatos = pd.read_excel(settings.DEPUTADOS_CONTATOS_XLSX_IN)
df_contatos['Nome Civil'] = df_contatos['Nome Civil'].str.title()

df_contatos.head()

,Nome Civil,Partido,E-mail,Telefone,Endereço,Data de Nascimento,Naturalidade
0,Adail José Figueiredo Pinheiro,REPUBLICANOS - AM,dep.adailfilho@camara.leg.br,(61) 3215-5531,Gabinete 531 - Anexo IV - Câmara dos Deputados,16/02/1992,Manaus - AM
1,Adilson Barroso Oliveira,PL - SP,dep.adilsonbarroso@camara.leg.br,(61) 3215-5603,Gabinete 603 - Anexo IV - Câmara dos Deputados,14/06/1964,Minas Novas - MG
2,Adolfo Viana De Castro Neto,PSDB - BA,dep.adolfoviana@camara.leg.br,(61) 3215-5911,Gabinete 911 - Anexo IV - Câmara dos Deputados,02/02/1981,Salvador - BA
3,Adriana Miguel Ventura,NOVO - SP,dep.adrianaventura@camara.leg.br,(61) 3215-5802,Gabinete 802 - Anexo IV - Câmara dos Deputados,06/03/1969,São Paulo - SP
4,Adriano Antônio Avelar,PP - GO,dep.adrianodobaldy@camara.leg.br,(61) 3215-5419,Gabinete 419 - Anexo IV - Câmara dos Deputados,06/09/1969,Goiás - GO


In [24]:
# 2. Limpar espaços em branco nos nomes para garantir o cruzamento correto
df_insta['Nome Civil'] = df_insta['Nome Civil'].str.strip()
df_contatos['Nome Civil'] = df_contatos['Nome Civil'].str.strip()

# 3. Selecionar apenas as colunas necessárias da planilha que tem o Instagram
# Usamos 'Nome Civil' para o cruzamento e 'Instagram' como o dado que queremos trazer
df_subset_insta = df_insta[['Nome Civil', 'Instagram']].drop_duplicates(subset='Nome Civil')

# 4. Realizar o merge (união) do tipo 'left'
# Isso mantém todos os dados da planilha de contatos e adiciona o Instagram onde houver correspondência
df_contatos_insta = pd.merge(df_contatos, df_subset_insta, on='Nome Civil', how='left')

df_contatos_insta.head()

,Nome Civil,Partido,E-mail,Telefone,Endereço,Data de Nascimento,Naturalidade,Instagram
0,Adail José Figueiredo Pinheiro,REPUBLICANOS - AM,dep.adailfilho@camara.leg.br,(61) 3215-5531,Gabinete 531 - Anexo IV - Câmara dos Deputados,16/02/1992,Manaus - AM,https://www.instagram.com/adailfilho/
1,Adilson Barroso Oliveira,PL - SP,dep.adilsonbarroso@camara.leg.br,(61) 3215-5603,Gabinete 603 - Anexo IV - Câmara dos Deputados,14/06/1964,Minas Novas - MG,https://www.instagram.com/adilson_barroso_ambi...
2,Adolfo Viana De Castro Neto,PSDB - BA,dep.adolfoviana@camara.leg.br,(61) 3215-5911,Gabinete 911 - Anexo IV - Câmara dos Deputados,02/02/1981,Salvador - BA,https://www.instagram.com/adolfovianaoficial/
3,Adriana Miguel Ventura,NOVO - SP,dep.adrianaventura@camara.leg.br,(61) 3215-5802,Gabinete 802 - Anexo IV - Câmara dos Deputados,06/03/1969,São Paulo - SP,https://www.instagram.com/adriventurasp/
4,Adriano Antônio Avelar,PP - GO,dep.adrianodobaldy@camara.leg.br,(61) 3215-5419,Gabinete 419 - Anexo IV - Câmara dos Deputados,06/09/1969,Goiás - GO,https://www.instagram.com/adrianodobaldyoficial/


In [5]:
dados_dict = {}

# 1. Usar 'with open' para abrir e fechar o arquivo automaticamente
# 'r' significa modo de leitura (read)
try:
    with open(settings.APIFY_JSON_OUT, 'r', encoding='utf-8') as arquivo:
        # 2. Usar json.load() para ler o arquivo e converter para dict
        dados_dict = json.load(arquivo)

    df_deputados = pd.read_json(settings.APIFY_JSON_OUT, orient='records')

except FileNotFoundError:
    print(f"Erro: O arquivo '{settings.APIFY_CSV_OUT}' não foi encontrado.")
except json.JSONDecodeError:
    print(f"Erro: O arquivo '{settings.APIFY_CSV_OUT}' não é um JSON válido.")

In [6]:
list_of_post_dataframes = []

for registro in dados_dict:
    # Check if the key exists AND if the value is not empty (for safety)
    if 'latestPosts' in registro and registro['latestPosts']:
        
        posts = registro['latestPosts']
        
        for post in posts:
            post['inputUrl'] = registro['inputUrl']
        
        df_posts = pd.DataFrame(posts)
        list_of_post_dataframes.append(df_posts)

# 1. Concatenate all DataFrames in the list into one master DataFrame
# 'ignore_index=True' is used to reset the index of the resulting DataFrame
df_posts_total = pd.concat(list_of_post_dataframes, ignore_index=True)

# 2. Convert the 'timestamp' column to datetime objects
# Assuming 'timestamp' is in seconds (a common format for API timestamps)
df_posts_total['data_datetime'] = pd.to_datetime(df_posts_total['timestamp'])

In [7]:
df_posts_agrupado = df_posts_total.groupby('inputUrl').agg(
    commentsCount=('commentsCount', 'sum'),
    likesCount=('likesCount', 'sum'),
    videoViewCount=('videoViewCount', 'sum'),
    post_count=('id', 'count'),
    data_datetime_max=('data_datetime', 'max'),
    data_datetime_min=('data_datetime', 'min'),
).reset_index()

df_deputados_join = pd.merge(df_deputados, df_posts_agrupado, how='left', on='inputUrl')

df_deputados_join['% Engajamento'] = (df_deputados_join['likesCount'] + df_deputados_join['commentsCount']) / df_deputados_join['followersCount']

df_deputados_join['% commentsCount'] = df_deputados_join['commentsCount'] / df_deputados_join['followersCount']

df_deputados_join['% likesCount'] = df_deputados_join['likesCount'] / df_deputados_join['followersCount']

df_deputados_join['likesCount\Posts'] = df_deputados_join['likesCount'] / df_deputados_join['post_count']

df_deputados_join['commentsCount\Posts'] = df_deputados_join['commentsCount'] / df_deputados_join['post_count']

# 1. Calcular o período em dias que os posts de cada deputado cobrem
periodo_dias = (df_deputados_join['data_datetime_max'] - df_deputados_join['data_datetime_min']).dt.days

# 2. Calcular a frequência (ex: posts por dia)
# Adicionamos +1 para evitar divisão por zero se todos os posts foram no mesmo dia
df_deputados_join['Frequencia (Posts/Dia)'] = df_deputados_join['post_count'] / (periodo_dias + 1)

# Ou, se preferir (dias por post):
df_deputados_join['Frequencia (Dias/Post)'] = periodo_dias / df_deputados_join['post_count']

In [8]:
df_deputados.columns

Index(['inputUrl', 'id', 'username', 'url', 'fullName', 'biography',
       'externalUrls', 'externalUrl', 'externalUrlShimmed', 'followersCount',
       'followsCount', 'hasChannel', 'highlightReelCount', 'isBusinessAccount',
       'joinedRecently', 'businessCategoryName', 'private', 'verified',
       'profilePicUrl', 'profilePicUrlHD', 'igtvVideoCount', 'relatedProfiles',
       'latestIgtvVideos', 'postsCount', 'latestPosts', 'fbid',
       'businessAddress', 'error', 'errorDescription', 'isRestrictedProfile',
       'restrictionReason'],
      dtype='str')

In [9]:
df_posts_total.columns

Index(['id', 'type', 'shortCode', 'caption', 'hashtags', 'mentions', 'url',
       'commentsCount', 'dimensionsHeight', 'dimensionsWidth', 'displayUrl',
       'images', 'videoUrl', 'alt', 'likesCount', 'videoViewCount',
       'timestamp', 'childPosts', 'ownerUsername', 'ownerId', 'productType',
       'isCommentsDisabled', 'inputUrl', 'taggedUsers', 'locationName',
       'locationId', 'musicInfo', 'isPinned', 'data_datetime'],
      dtype='str')

In [10]:
df_deputados_join.columns

Index(['inputUrl', 'id', 'username', 'url', 'fullName', 'biography',
       'externalUrls', 'externalUrl', 'externalUrlShimmed', 'followersCount',
       'followsCount', 'hasChannel', 'highlightReelCount', 'isBusinessAccount',
       'joinedRecently', 'businessCategoryName', 'private', 'verified',
       'profilePicUrl', 'profilePicUrlHD', 'igtvVideoCount', 'relatedProfiles',
       'latestIgtvVideos', 'postsCount', 'latestPosts', 'fbid',
       'businessAddress', 'error', 'errorDescription', 'isRestrictedProfile',
       'restrictionReason', 'commentsCount', 'likesCount', 'videoViewCount',
       'post_count', 'data_datetime_max', 'data_datetime_min', '% Engajamento',
       '% commentsCount', '% likesCount', 'likesCount\Posts',
       'commentsCount\Posts', 'Frequencia (Posts/Dia)',
       'Frequencia (Dias/Post)'],
      dtype='str')

# Segmentação

In [27]:
metrics = [
    '% Engajamento',
    '% likesCount',
    '% commentsCount',
    'likesCount\\Posts',
    'commentsCount\\Posts',
    'Frequencia (Posts/Dia)',
    'followersCount',
]

top_n = 5

for metric in metrics:
    
    # Filtrar e ordenar como no código original
    df_piores = (
        df_deputados_join[df_deputados_join[metric].notna()]
        .sort_values(metric, ascending=True)
        .head(top_n)
    )

    print(f"\nTop {top_n} piores deputados para a métrica '{metric}':")
    print(df_piores[['fullName', metric]])

    set_piores_inputUrl = set()

    # Adicionar os inputUrl ao set
    set_piores_inputUrl.update(df_piores['inputUrl'].dropna())

    # Filtrar o DataFrame do Excel
    df_contatos_filtrados = df_contatos_insta[df_contatos_insta['Instagram'].isin(set_piores_inputUrl)]

    safe_metric = metric.replace('\\', '-').replace('/', '-')
    output_path = settings.PIORES_RANKINGS_DIR / f'top {top_n} piores {safe_metric}.xlsx'

    df_contatos_filtrados.to_excel(output_path, index=False)
    print(f"Arquivo salvo em: {output_path}")


Top 5 piores deputados para a métrica '% Engajamento':
                                  fullName  % Engajamento
119                             Leo Prates           0.01
210                            Vitor Lippi           0.01
194                       Pompeo de Mattos           0.02
214  Vinicius Carvalho | A Serviço do Povo           0.02
261                           Silvye Alves           0.02
Arquivo salvo em: c:\Projetos\analise-deputados-brasil\data\processed\piores_rankings\top 5 piores % Engajamento.xlsx

Top 5 piores deputados para a métrica '% likesCount':
                   fullName  % likesCount
119              Leo Prates          0.01
210             Vitor Lippi          0.01
261            Silvye Alves          0.02
194        Pompeo de Mattos          0.02
191  Dep. Renilce Nicodemos          0.02
Arquivo salvo em: c:\Projetos\analise-deputados-brasil\data\processed\piores_rankings\top 5 piores % likesCount.xlsx

Top 5 piores deputados para a métrica '% commentsCoun

In [12]:
df_contatos_filtrados

,Nome Civil,Partido,E-mail,Telefone,Endereço,Data de Nascimento,Naturalidade,Instagram


In [51]:
# Debug: Verificar correspondências em Nome Civil
print("Exemplos de Nome Civil em df_deputados:")
print(df_deputados['Nome Civil'].head())

print("\nExemplos de Nome Civil em df_insta:")
print(df_insta['Nome Civil'].head())

print("\nExemplos de Nome Civil em df_contatos:")
print(df_contatos['Nome Civil'].head())

print("\nVerificar interseção de Nome Civil entre df_deputados e df_insta:")
intersecao_nomes = set(df_deputados['Nome Civil'].dropna()) & set(df_insta['Nome Civil'].dropna())
print("Interseção:", len(intersecao_nomes), "nomes")

print("\nVerificar interseção de Nome Civil entre df_contatos e df_insta:")
intersecao_nomes2 = set(df_contatos['Nome Civil'].dropna()) & set(df_insta['Nome Civil'].dropna())
print("Interseção:", len(intersecao_nomes2), "nomes")

# Verificar se os nomes em df_piores estão em df_contatos_insta
print("\nNomes em df_piores:")
piores_nomes = df_piores['Nome Civil'].dropna()
print(piores_nomes)

print("\nEstes nomes estão em df_contatos_insta?")
for nome in piores_nomes:
    esta_presente = nome in df_contatos_insta['Nome Civil'].values
    print(f"{nome}: {esta_presente}")

Exemplos de Nome Civil em df_deputados:


KeyError: 'Nome Civil'